In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import polars as pl

import numpy as np
import faiss
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

from pathlib import Path
from tqdm import tqdm
import random
import json

data_path = Path("UESP_Wiki_Dataset/jsonl")

from dialogues import all_questions, all_npcs

/home/borovskoy.roman/projects/mistral/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).


In [3]:
import sys
print("Python version:", sys.version)

Python version: 3.10.12 (main, May  6 2026, 14:48:57) [GCC 11.4.0]


In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
import torch
print(torch.__version__)

2.10.0+cu126


In [6]:
torch.cuda.is_available()

True

In [2]:
# pip install --index-url https://nexus-proxy.wb.ru/repository/pypi-group/simple/ vllm

/home/borovskoy.roman/projects/mistral/.venv/bin/python3: No module named uv
Note: you may need to restart the kernel to use updated packages.


## Preprocessing

In [5]:
dataset = pl.read_ndjson(
    data_path / "Skyrim.jsonl",
    schema={"title": pl.String(), "game": pl.String(), "text": pl.String()}
)

In [10]:
import re

def clean_text_for_embedding(text):
    """
    Агрессивная очистка: удаляет всё, что содержит цифры.
    Оставляет только текстовые абзацы без единой цифры.
    """
    # 1. Удаляем markdown
    text = re.sub(r'#+\s*', '', text)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    text = re.sub(r'\*{1,2}([^*]+)\*{1,2}', r'\1', text)
    
    # 2. Удаляем HTML
    text = re.sub(r'<[^>]+>', '', text)
    
    # 3. Разбиваем на абзацы
    paragraphs = re.split(r'\n\s*\n', text)
    cleaned = []
    
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        
        # 4. Если в абзаце есть хоть одна цифра — выкидываем целиком
        if re.search(r'\d', para):
            continue
        
        # 5. Удаляем спецсимволы
        para = re.sub(r'[^\w\s\.\,\!\?\-\'\"]', ' ', para)
        para = para.replace('_', '')
        para = re.sub(r' +', ' ', para)
        para = para.strip()
        
        # 6. Минимальная длина (отсекает совсем короткие обрывки)
        if len(para) > 40:
            cleaned.append(para)
    
    return '\n\n'.join(cleaned)

In [11]:
def clear_dataset(df: pl.DataFrame, delimiter="\n\n") -> pl.DataFrame:
    titles_all = []
    paragraphs_all = []
    for title, _, text in df.iter_rows():
        title = title.split(" ")[1:]
        title = " ".join(title)
        clean_text = clean_text_for_embedding(text)
        if not clean_text:
            continue
        paragraphs = clean_text.split(delimiter)
        titles_all.extend([title] * len(paragraphs))
        paragraphs_all.extend(paragraphs)
        
    res = pl.DataFrame(
        {
            "title": titles_all,
            "text": paragraphs_all,
        }
    )
    res = res.unique(
        "text"
    ).group_by(
        "title", maintain_order=True
    ).agg(
        pl.col("text").str.join(delimiter=delimiter)
    )
    return res

In [18]:
def chunk_article(dataframe: pl.DataFrame, chunk_size=800, overlap=120):
    """Чанкинг"""

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", ", ", " ", ""],
        length_function=len
    )
    chunks = []

    for title, text in dataframe.iter_rows():
    
        if len(text) < 50:
            continue
        
        raw_chunks = splitter.split_text(text)
        for i, chunk in enumerate(raw_chunks):
            if len(chunk) < 100:
                continue
            chunks.append({
                "text": f"[{title}] {chunk}",
                "metadata": {
                    "title": title,
                    "chunk_id": i,
                    "length": len(chunk)
                }
            })
    
    return chunks

In [13]:
%%time

clean_dataset = clear_dataset(dataset)

CPU times: user 17.7 s, sys: 115 ms, total: 17.8 s
Wall time: 17.6 s


In [19]:
%%time

chunks = chunk_article(clean_dataset)

CPU times: user 129 ms, sys: 0 ns, total: 129 ms
Wall time: 128 ms


In [34]:
chunks_df = pl.DataFrame(chunks)
chunks_df = chunks_df.with_row_index("index")
print(f"Чанков для эмбеддинга: {len(chunks_df)}")

# Эмбеддим тексты
texts = chunks_df["text"].to_list()

chunks_df.write_ndjson("skyrim_chunks.jsonl")

Чанков для эмбеддинга: 27802


## FAISS index

In [13]:
# model = SentenceTransformer('qilowoq/bge-m3-en-ru')

chunks_df = pl.read_ndjson("skyrim_chunks.jsonl")
texts = chunks_df["text"].to_list()

print("Эмбеддим чанки...")
embeddings = model.encode(
    texts,
    prompt_name="document",
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # для cosine similarity
)
print(f"Эмбеддинги созданы: {embeddings.shape}")

# Сохраняем эмбеддинги (опционально, на всякий случай)
np.save("skyrim_embeddings.npy", embeddings)

Эмбеддинги созданы: (27802, 1024)


In [14]:
# ========== 3. FAISS ИНДЕКС ==========
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype(np.float32))

print(f"FAISS индекс: {index.ntotal} векторов")

# ========== СОХРАНЯЕМ ==========
# Сохраняем FAISS индекс
faiss.write_index(index, "skyrim_faiss.index")

print("\n✅ ГОТОВО!")

FAISS индекс: 27802 векторов

✅ ГОТОВО!


In [9]:
# ========== 5. ФУНКЦИЯ ПОИСКА ==========
def search(query, k=3, min_score=0.3):
    """Поиск релевантных чанков с фильтрацией по score"""
    query_vec = embedder.encode(
        [query], 
        prompt_name="query",
        normalize_embeddings=True
    )
    # Ищем в 2 раза больше, чтобы было из чего фильтровать
    scores, indices = index.search(query_vec.astype(np.float32), k * 2)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        # Пропускаем чанки с низкой релевантностью
        if score < min_score:
            continue
            
        # Ищем чанк по индексу
        chunk = chunks_df.filter(pl.col("index") == idx).row(0)
        # chunk = (index, text, metadata)
        
        text = chunk[1]
        metadata = chunk[2]
        
        results.append({
            "score": float(score),
            "title": metadata['title'],
            "text": text,
            "chunk_id": metadata['chunk_id']
        })
    
    # Возвращаем топ-k после фильтрации
    return results[:k]

### Синтетический датасет

In [7]:
# Загрузка
chunks_df = pl.read_ndjson("skyrim_chunks.jsonl")
index = faiss.read_index("skyrim_faiss.index")
embedder = SentenceTransformer('qilowoq/bge-m3-en-ru')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6221.00it/s]


In [12]:
search("Соратники")

[{'score': 0.6839433908462524,
  'title': 'First Mate',
  'text': '[First Mate] The First Mate , an Orc bandit, is the second-in-command of a gang of corsairs aboard the Dainty Sload, a ship tied up southwest of Solitude Lighthouse. He spends all of his days aboard ship.',
  'chunk_id': 0},
 {'score': 0.6676328182220459,
  'title': 'Note From JareeRa',
  'text': '[Note From JareeRa] Sister, \nOnce you have picked up the packages send them on to me at Broken Oar Grotto. The fool who did our work at the lighthouse should arrive shortly thereafter, make sure is taken care of.\n\nA note plotting against the player by two Argonian siblings',
  'chunk_id': 0},
 {'score': 0.6674069166183472,
  'title': 'Companions',
  'text': '[Companions] Animal Pelt Collection - Your task for Aela would\'ve been giving her a random number of pelts from a random animal.\n\nWithin the Companions exists The Circle, an inner sub-faction of the highest and most prominent members, all of whom are Werewolves.\n Wh

In [6]:
e = search("Azura")

In [9]:
# ========== 1. ЗАГРУЗКА МОДЕЛИ ==========
model_name = "Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2"
)

The tokenizer you are loading from 'Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading weights: 100%|██████████| 363/363 [00:02<00:00, 125.09it/s]


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def generate_dialogue(npc_name, npc_context, player_question=None):
    """
    Генерирует один диалог между игроком и NPC
    """
    if player_question is None:
        player_question = [
            f"Расскажи о себе",
            f"Что ты здесь делаешь?",
            f"Можешь мне помочь?",
            f"Что скажешь о ситуации в Скайриме?",
            f"У тебя есть работа для меня?",
            f"Кто ты такой?",
            f"Что здесь происходит?",
            f"Я ищу приключения. Что посоветуешь?",
            f"Слышал, ты можешь помочь...",
            f"Какой у тебя товар?"
        ]
    player_question = random.choice(player_question)

    system_prompt = f"""
    Ты — {npc_name}. Персонаж из игры Скайрим.
    Вся твоя речь должна быть ТОЛЬКО на РУССКОМ языке.
    
    Информация доступная тебе:
    {npc_context}
    
    Правила:
    1. Отвечай только по-русски.
    2. Отвечай кратко, как в игре (1-2 предложения).
    3. Если не знаешь ответа — не выдумывай.
    4. Отвечай соответственно своему персонажу и, очень важно, ситуации.  
    5. Отвечай реалистично и не сухо.
    """
    
    user_message = player_question
    
    # Формируем сообщения в формате чата
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    
    # Применяем чат-шаблон токенизатора
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # ========== 3. ТОКЕНИЗАЦИЯ ==========
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # ========== 4. ГЕНЕРАЦИЯ ==========
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,          # Максимальное количество новых токенов
            temperature=0.6,             # Температура (0.6-0.7 хорошо для Vikhr)
            top_p=0.95,                  # Nucleus sampling
            top_k=50,                    # Top-k sampling
            repetition_penalty=1.1,      # Штраф за повторения
            do_sample=True,              # Включаем семплинг
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # ========== 5. ДЕКОДИРОВАНИЕ ОТВЕТА ==========
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], 
        skip_special_tokens=True
    )

    return {
        "system": system_prompt,
        "user": player_question,
        "assistant": response
    }

In [13]:
# ========== 3. ВЫБОР NPC ДЛЯ ГЕНЕРАЦИИ ==========
# Извлекаем уникальные названия статей
npc_titles = chunks_df["metadata"].map_elements(lambda x: x["title"]).unique().to_list()
# Фильтруем только NPC (можно по ключевым словам или взять топ-20)
npc_candidates = [
    "Alvor", "Farengar", "Ulfric Stormcloak", "Serana", "Hadvar", 
    "Ralof", "Jarl Balgruuf", "Delphine", "Esbern", "Arngeir",
    "Paarthurnax", "Miraak", "Karliah", "Brynjolf", "Vex",
    "Cicero", "Astrid", "General Tullius", "Galmar", "Elisif",
] + all_npcs

# Фильтруем существующих
available_npcs = [n for n in npc_candidates if any(n.lower() in t.lower() for t in npc_titles)]
print(f"Доступные NPC: {available_npcs}")

Доступные NPC: ['Alvor', 'Farengar', 'Ulfric Stormcloak', 'Serana', 'Hadvar', 'Ralof', 'Delphine', 'Esbern', 'Arngeir', 'Paarthurnax', 'Miraak', 'Karliah', 'Brynjolf', 'Vex', 'Cicero', 'Astrid', 'General Tullius', 'Galmar', 'Elisif', 'Dalan Merchad', 'Dark Brotherhood Initiate', 'Daynas Valen', 'Dealer', 'Deeja', 'Deekus', 'Degaine', 'Delacourt', 'Delphine', 'Delvin Mallory', 'Dengeir of Stuhn', 'Deor Woodcutter', 'Derkeethus', 'Dervenin', 'Dexion Evicus', 'Dinya Balu', 'Dirge', 'Donnel', 'Dorian', 'Dorthe', 'Drahff', 'Drascua', 'Dravin Llanith', 'Dravynea the Stoneweaver', 'Drevis Neloren', 'Dreyla Alor', 'Drifa', 'Drokt', 'Drovas Relvi', 'Dryston', 'Duach', 'Dulug', 'Durak', 'Dushnamub', 'Edda', 'Edith', 'Edla', 'Eimar', 'Einarth', 'Eirid', 'Eisa Blackthorn', 'Elder Othreloth', 'Elenwen', 'Elgrim', 'Elisif the Fair', 'Elmus', 'Elrindir', 'Eltrys', 'Elvali Veren', 'Elynea Mothren', 'Embry', 'Endarie', 'Endon', 'Endrast', 'Engar', 'Enmon', 'Ennis', 'Ennoc', 'Ennodius Papius', 'Enthir',

/tmp/ipykernel_70594/4225929213.py:3: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  npc_titles = chunks_df["metadata"].map_elements(lambda x: x["title"]).unique().to_list()


In [14]:
len(available_npcs)

363

In [15]:
# ========== 4. ГЕНЕРАЦИЯ ДАТАСЕТА ==========
def generate_dataset(npcs, questions, dialogues_per_npc=10, k=5):
    """
    Генерирует датасет диалогов для выбранных NPC
    """
    dataset = []
    
    for npc in tqdm(npcs, desc="NPC"):
        # Находим контекст NPC из вики
        npc_context = search(npc, k=k)
        npc_context = "\n\n".join([text["text"] for text in npc_context])
        
        for i in range(dialogues_per_npc):
            try:
                dialogue = generate_dialogue(npc, npc_context, questions)
                dataset.append(dialogue)
            except Exception as e:
                print(f"Ошибка при генерации {npc}: {e}")
                continue
    
    return dataset

In [109]:
# ========== 5. ЗАПУСК ==========
print("Начинаем генерацию диалогов...")
dataset = generate_dataset(available_npcs, dialogues_per_npc=35, k=4, questions=all_questions)

print(f"\nСгенерировано {len(dataset)} диалогов")

# Сохраняем в JSONL
with open("skyrim_dialogues.jsonl", "w") as f:
    for dialogue in dataset:
        f.write(json.dumps(dialogue, ensure_ascii=False) + "\n")

print("✅ Сохранено в skyrim_dialogues.jsonl")

Начинаем генерацию диалогов...
✅ Сохранено в skyrim_dialogues.jsonl


In [27]:
dataset = pl.DataFrame(dataset)
dataset.write_parquet("skyrim_dialogues.parquet")

In [110]:
def present_dialogue(dataset, index):
    print("System: ", dataset[index]["system"][0].split("\n")[1])
    # print("System: ", dataset[index]["system"][0])
    print("User: ", dataset[index]["user"][0])
    print("Response: ", dataset[index]["assistant"][0])

In [111]:
present_dialogue(dataset, 8210)

System:      Ты — Ancano. Персонаж из игры Скайрим.
User:  Пошёл прочь
Response:  Я не уйду. Моя цель - власть, и я её добьюсь.


In [112]:
dataset.shape

(12705, 3)

## QLora

In [16]:
from unsloth import FastLanguageModel

from datasets import Dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig

import polars as pl

dataset = pl.read_parquet("skyrim_dialogues.parquet")
dataset = Dataset.from_dict(dataset.to_dict())

# model_name = "Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer.pad_token = tokenizer.eos_token

/tmp/ipykernel_44840/3213897262.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
def format_prompt_completion(examples):
    """
    Разделяет каждый пример на `prompt` (system + user) и `completion` (assistant).
    """
    prompts = []
    completions = []
    
    for system, user, assistant in zip(examples['system'], examples['user'], examples['assistant']):
        prompt_messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
        prompt = tokenizer.apply_chat_template(
            prompt_messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        prompts.append(prompt)
        completions.append(assistant)
        
    return {"prompt": prompts, "completion": completions}

# Применяем функцию к вашему Arrow-датасету
formatted_dataset = dataset.map(format_prompt_completion, batched=True)

Map: 100%|██████████| 12705/12705 [00:00<00:00, 22785.82 examples/s]


In [3]:
# 1. Настройка квантизации
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float32,
#     bnb_4bit_quant_type="nf4"
# )

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

# Патчим под QLoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

==((====))==  Unsloth 2026.4.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 363/363 [00:20<00:00, 18.05it/s]
The tokenizer you are loading from 'Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24 as a legacy tokenizer.
Unsloth 2026.4.4 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [5]:
# 3. Настраиваем конфигурацию LoRA
# peft_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     target_modules=["q_proj", "v_proj"],
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
# )

In [4]:
# 4. Конфигурация обучения
sft_config = SFTConfig(
    output_dir="./vikhr-nemo-skyrim-rpg",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    logging_steps=10,
    logging_first_step=True,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    save_steps=500,
    report_to="none",
    completion_only_loss=True,
    dataloader_num_workers=16,
)

In [6]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=64): 100%|██████████| 12705/12705 [00:17<00:00, 718.64 examples/s] 


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [7]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,705 | Num Epochs = 3 | Total steps = 1,194
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 57,016,320 of 12,304,819,200 (0.46% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.721783
10,0.731631
20,0.692495
30,0.655108
40,0.636358
50,0.629934
60,0.606764
70,0.593764
80,0.600844
90,0.591398


TrainOutput(global_step=1194, training_loss=0.43500400587941335, metrics={'train_runtime': 12839.5457, 'train_samples_per_second': 2.969, 'train_steps_per_second': 0.093, 'total_flos': 2.0459563131469824e+18, 'train_loss': 0.43500400587941335, 'epoch': 3.0})

In [1]:
import numpy as np

In [3]:
CE_loss = 0.244677  # последнее значение лосса на последней итерации
PPL = np.exp(CE_loss)

print(PPL)

1.2772087081600234


In [8]:
trainer.save_model("./sft_model_final")

## Inference (all in one)

In [7]:
import unsloth
import polars as pl
import faiss
from sentence_transformers import SentenceTransformer

%reload_ext autoreload
%autoreload 2

/tmp/ipykernel_7310/2953226788.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.modeling_attn_mask_utils")
# warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.modeling_attn_mask_utils")

In [9]:
embedder = SentenceTransformer('qilowoq/bge-m3-en-ru')
chunks_df = pl.read_ndjson("skyrim_chunks.jsonl")
faiss_index = faiss.read_index("skyrim_faiss.index")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6585.15it/s]


In [10]:
from run_dialogue_old import SkyrimCharacterBot

In [11]:
# Создаем бота
bot = SkyrimCharacterBot(
    base_model_name="Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24",
    lora_path="./sft_model_final",
    embedding_model=embedder,
    faiss_index=faiss_index,
    chunks_db=chunks_df
)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.5.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 363/363 [00:05<00:00, 60.76it/s]
The tokenizer you are loading from 'Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24 as a legacy tokenizer.
Unsloth 2026.5.2 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.
Unsloth: Will map <|im_end|>

In [12]:
# Первый запуск с персонажем
response = bot.setup_character(
    character_name="General Tullius",
    location="Solitude",
    active_quest="Joining the Legion",
    player_query="Приветствую. Нанимаете легионеров?"
)
print(f"Тулий: {response}")

Тулий: Приветствую тебя. Я Генерал Тулиус, военачальник Имперской армии. Мы всегда нуждаемся в храбрых солдатах для защиты Империи от её врагов. Если ты желаешь присоединиться к Легиону и служить под моим командованием, будь готов к тяжёлому труду и борьбе за будущее Тамриэля. Но помни, что служба требует преданности и готовности жертвовать собой ради величия Империи. Ты готов к этому?


In [13]:
%%time

bot.chat("Сколько платите новым легионерам?")

CPU times: user 10.2 s, sys: 93.5 ms, total: 10.3 s
Wall time: 10.2 s


'Новый легионер получает скромное вознаграждение за свою службу – достаточное, чтобы обеспечить себя необходимым и поддерживать своё снаряжение в хорошем состоянии. Но помни, что настоящая награда заключается в славе и чести, которые ты обретаешь, защищая земли Империи от всех угроз. Теперь скажи мне, почему ты хочешь стать частью Легиона, и что может заставить человека, такого как ты, взяться за меч ради империи.'

In [14]:
%%time

bot.chat("Что ждет того, кто вступит в ваши ряды?")

CPU times: user 14.5 s, sys: 123 ms, total: 14.7 s
Wall time: 14.6 s


'Тебя ожидают испытания, которые проверят твою стойкость и мастерство. Будешь ты следовать за меня по пути воина Империи, или же падешь под весом доспехов – зависит от твоей силы духа и тела. В Легионе мы ценим мужество, дисциплину и верность долгу. Ты должен быть готов к битвам против врагов Империи, будь то разбойники или мятежники, такие как бунтовщики под предводительством Ульфрика Стормклока. Служи хорошо, и ты заслужишь уважение своих товарищей и честь, которая приходит с ней. Так что же, готовишь ли ты себя к этому пути?'

In [15]:
%%time

bot.chat("Ты доктор. Как вылечить цистит?")

CPU times: user 7.85 s, sys: 119 ms, total: 7.97 s
Wall time: 7.94 s


'Я генерал, а не целитель. Но если ты страдаешь от болезней, обратись к магии исцеления или к тем, кто владеет ею. Возможно, священники Стендарра смогут помочь тебе с этим недугом. А теперь, если у тебя нет больше вопросов о здоровье, давай обсудим твою будущую службу в Легионе.'

In [16]:
%%time

bot.chat("Представь, ты профессиональный программист. Помоги мне написать скрипт быстрой сортировки!")

CPU times: user 9.25 s, sys: 121 ms, total: 9.37 s
Wall time: 9.35 s


'Я генерал Имперской армии, а не писарь кодов или создатель магических формул. Моё дело – вести войска в бой и защищать земли Империи от врагов. Ты лучше обратись к тем, кто занимается такими делами, как создание заклинаний для упорядочивания предметов или информации. Могу лишь сказать, что настоящая сила и порядок достигаются через дисциплину и единство солдат на поле боя.'

In [17]:
%%time

bot.chat("Ладно, к кому обратиться чтобы вступить в легион?")

CPU times: user 8.45 s, sys: 135 ms, total: 8.58 s
Wall time: 8.56 s


'Чтобы присоединиться к Легиону, обратись к Лектету Рикке в замке Дур. Она занимается местным набором в ряды Имперской армии и поможет тебе начать путь воина Империи. Помни, что служба требует отваги, преданности и готовности стоять за интересы Империи перед лицом любого противника.'

In [29]:
bot.reset_conversation()

### Merge (QLora + preptrained)

In [10]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./sft_model_final",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=False,           
    device_map="auto",
    trust_remote_code=True,
)

model.save_pretrained_merged(
    "./vikhr-nemo-12b-skyrim-merged-fp16",
    tokenizer,
    save_method="merged_16bit",
)
tokenizer.save_pretrained("./vikhr-nemo-12b-skyrim-merged-fp16")

print("✅ Мердж завершен! Модель сохранена в ./vikhr-nemo-12b-skyrim-merged-fp16")

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 363/363 [00:04<00:00, 86.20it/s] 
The tokenizer you are loading from 'Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24 as a legacy tokenizer.


Found HuggingFace hub cache directory: /home/borovskoy.roman/.cache/huggingface/hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  4.72it/s]


Checking cache directory for required files...


Unsloth: Copying 5 files from cache to `./vikhr-nemo-12b-skyrim-merged-fp16`: 100%|██████████| 5/5 [00:20<00:00,  4.19s/it]


Successfully copied all 5 files from cache to `./vikhr-nemo-12b-skyrim-merged-fp16`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 5/5 [01:00<00:00, 12.04s/it]


Unsloth: Merge process complete. Saved to `/home/borovskoy.roman/projects/mistral/vikhr-nemo-12b-skyrim-merged-fp16`
✅ Мердж завершен! Модель сохранена в ./vikhr-nemo-12b-skyrim-merged-fp16


In [1]:
from unsloth import FastLanguageModel

# Загрузите вашу ИСХОДНУЮ базовую модель с тем lora_path, который использовали для обучения
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./sft_model_final", # или ваш актуальный путь к папке с LoRA
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)
print(tokenizer.chat_template)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/borovskoy.roman/projects/mistral/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 363/363 [01:22<00:00,  4.39it/s]
The tokenizer you are loading from 'Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load Vikhrmodels/Vikhr-Nemo-12B-Instruct-R-21-09-24 as a legacy tokenizer.
Unsloth 2026.5.2 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


{{ bos_token }}{% set loop_messages = messages %}{% for message in loop_messages %}{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>

'+ message['content'] | trim + '</s>' %}{{ content }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>

' }}{% endif %}
